In [ ]:
import math
import random
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

from pytorch_metric_learning.losses import CosFaceLoss, CircleLoss
from transformers import ConvNextV2ForImageClassification

from lib.muon import Muon
from src.utils import kfold_split, query_gallery_split
from src.metrics import evaluate

In [ ]:
torch._dynamo.config.recompile_limit = 16

In [ ]:
seed = 56
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
class RandomDataset(torch.utils.data.Dataset):
    def __init__(self, root_path, data, transform=None):
        self.root_path = root_path
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        label, camera = self.data.loc[index, ["label", "camera_id"]]

        same_vehicle_other_camera = self.data[
            (self.data["label"] == label) & (self.data["camera_id"] != camera)
        ]
        assert len(same_vehicle_other_camera) > 0
        positive_sample = same_vehicle_other_camera.sample().index[0]

        other_vehicle_same_camera = self.data[
            (self.data["label"] != label) & (self.data["camera_id"] == camera)
        ]
        if len(other_vehicle_same_camera) > 0:
            negative_sample = other_vehicle_same_camera.sample().index[0]
            return (
                [
                    self.load_image(index),
                    self.load_image(positive_sample),
                    self.load_image(negative_sample),
                ],
                [label, label, self.data.loc[negative_sample, "label"]],
            )
        else:
            return (
                [
                    self.load_image(index),
                    self.load_image(positive_sample),
                ],
                [label, label],
            )

    def load_image(self, index):
        img_path = (
            self.root_path / self.data.loc[index, "image_id"]
        ).with_suffix(".jpg")
        assert img_path.is_file()
        image = torchvision.io.read_image(img_path)

        x, y, w, h = self.data.loc[index, ["x", "y", "w", "h"]]
        image = image[:, y : y + h, x : x + w]
        assert image.shape == (3, h, w)

        if self.transform is not None:
            image = self.transform(image)

        return image

In [ ]:
class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, root_path, data, transform=None):
        self.root_path = root_path
        self.data = data[["image_id", "x", "y", "w", "h"]].to_numpy()
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data[index]

        img_path = (self.root_path / row[0]).with_suffix(".jpg")
        assert img_path.is_file()
        image = torchvision.io.read_image(img_path)

        x, y, w, h = row[1:]
        image = image[:, y : y + h, x : x + w]
        assert image.shape == (3, h, w)

        if self.transform is not None:
            image = self.transform(image)

        return image

In [7]:
data = pd.read_csv("./data/train.csv")
data

,image_id,x,y,w,h,vehicle_id,camera_id
0,f0d68d754a8a444b8b49628b0633068c,1199,491,619,589,403,90
1,b94be87b5ced41f5868f2dfb415100e4,0,308,437,420,403,89
2,296ab330ff8e49dd878d922c10352ca0,1199,433,682,594,403,90
3,5b7baa07e07640449243f4619a3077cc,1,317,434,414,403,89
4,ee3240d083594b5dbed8d8d28c236f91,0,332,436,395,403,89
...,...,...,...,...,...,...,...
9551,33c775ad393847f180aa715ac2d4c970,631,201,822,626,330,48
9552,56b0fd4da91545038e7228e5fe3706f0,630,201,823,624,330,48
9553,24ab438a002044ceaf3ab0cfafb811e9,1014,35,713,516,330,73
9554,f15c0a764e054d5ebf103925ce5f13ec,1016,33,707,518,330,73


In [8]:
train_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224), antialias=True),
        transforms.RandomHorizontalFlip(),
        transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        transforms.RandomErasing(),
    ]
)
train_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    AutoAugment(policy=AutoAugmentPolicy.CIFAR10, fill=None)
    ConvertImageDtype()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0, inplace=False)
)

In [9]:
val_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224), antialias=True),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
)
val_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ConvertImageDtype()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
)

In [10]:
def collate_fn(batch):
    images, labels = zip(*batch)
    images = torch.stack(sum(images, []), dim=0)
    labels = torch.LongTensor(sum(labels, []))
    return images, labels

In [11]:
class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.backbone = ConvNextV2ForImageClassification.from_pretrained(
            "convnextv2-base-pretrained/", drop_path_rate=0.5, num_labels=0
        )
        self.proj = nn.Linear(1024, 512, bias=False)

    def forward(self, x):
        embeds = self.proj(self.backbone(pixel_values=x).logits)
        embeds = F.normalize(embeds, p=2, dim=-1)
        return embeds

In [12]:
folds = kfold_split(data, n_splits=3)
folds.append((data, None))

fold_preds = []
fold_maps = []

for idx, (train_df, val_df) in enumerate(tqdm(folds)):
    print(f"FOLD {idx}:")

    vehicle2label = {}
    for vehicle_id in train_df["vehicle_id"].unique():
        vehicle2label[vehicle_id] = len(vehicle2label)
    train_df["label"] = train_df["vehicle_id"].map(vehicle2label)
    num_classes = len(vehicle2label)

    train_dataset = RandomDataset(Path("data/images/"), train_df, train_transforms)

    train_dataloader = torch.utils.data.DataLoader(
        dataset=train_dataset,
        batch_size=16,
        num_workers=8,
        drop_last=True,
        shuffle=True,
        pin_memory=True,
        collate_fn=collate_fn,
        persistent_workers=True,
    )

    if val_df is not None:
        query_df, gallery_df = query_gallery_split(val_df)

        val_dataset = torch.utils.data.ConcatDataset(
            [
                InferenceDataset(Path("data/images/"), query_df, val_transforms),
                InferenceDataset(Path("data/images/"), gallery_df, val_transforms),
            ]
        )

        val_dataloader = torch.utils.data.DataLoader(
            dataset=val_dataset,
            batch_size=16,
            num_workers=8,
            drop_last=False,
            shuffle=False,
            pin_memory=True,
            persistent_workers=True,
        )

    model = Model()
    model.to(device)

    model.compile(mode="reduce-overhead", dynamic=False, fullgraph=True)

    main_loss_fn = CircleLoss(m=0.25, gamma=32)

    id_loss_fn = CosFaceLoss(
        num_classes=num_classes,
        embedding_size=512,
        margin=0.25,
        scale=math.sqrt(2) * math.log(num_classes - 1),
    )
    id_loss_fn.W.data.normal_(0.0, 0.02)

    muon_params = [
        p
        for name, p in model.named_parameters()
        if name.startswith("backbone.")
        and (
            name.rstrip("weight").endswith(
                ("pwconv1.", "pwconv2.", "downsampling_layer.1.")
            )
        )
    ]

    adam_params = [
        p
        for name, p in model.named_parameters()
        if name.startswith("backbone.")
        and not (
            name.rstrip("weight").endswith(
                ("pwconv1.", "pwconv2.", "downsampling_layer.1.")
            )
        )
    ]

    optimizer = Muon(
        [
            {"params": muon_params, "lr": 4e-5, "use_muon": True},
            {"params": adam_params, "lr": 4e-5, "use_muon": False},
            {"params": model.proj.weight, "lr": 1e-3, "use_muon": True},
            {"params": id_loss_fn.parameters(), "lr": 1e-3, "use_muon": False},
        ]
    )

    WARMUP_EPOCHS = 1
    EPOCHS = 17

    main_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0,
        end_factor=0.0,
        total_iters=EPOCHS * len(train_dataloader),
    )
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=0.01,
        end_factor=1.0,
        total_iters=WARMUP_EPOCHS * len(train_dataloader),
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, main_scheduler],
        milestones=[WARMUP_EPOCHS * len(train_dataloader)],
    )

    scores = []
    for epoch in tqdm(range(WARMUP_EPOCHS + EPOCHS)):
        print(f"EPOCH {epoch}:")

        model.train()

        running_loss = 0.0

        tbar = tqdm(total=len(train_dataloader), leave=False)
        for x, labels in train_dataloader:
            x = x.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                embeds = model(x)
            loss = main_loss_fn(embeds, labels) + id_loss_fn(embeds, labels)

            optimizer.zero_grad()
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            scheduler.step()

            running_loss += loss.detach()
            tbar.update(1)

        tbar.close()

        running_loss /= len(train_dataloader)
        print(f"Train Loss: {running_loss.item()}")

        if val_df is not None:
            model.eval()

            y_pred = []

            with torch.inference_mode():
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    for x in val_dataloader:
                        x = x.to(device, non_blocking=True)
                        embeds = model(x)
                        y_pred.append(embeds.clone())

            y_pred = torch.cat(y_pred).cpu()

            query_embeds = y_pred[: len(query_df)]
            gallery_embeds = y_pred[-len(gallery_df) :]

            s = (
                F.normalize(query_embeds, p=2, dim=-1)
                @ F.normalize(gallery_embeds, p=2, dim=-1).T
            )
            metric = evaluate(query_df, gallery_df, s.numpy())
            print("Val mAP:", metric)
            scores.append(metric)

    if scores:
        fold_preds.append((query_df, gallery_df, query_embeds, gallery_embeds))
        fold_maps.append(scores[-1])

np.mean(fold_maps), np.std(fold_maps)

  0%|          | 0/4 [00:00<?, ?it/s]

FOLD 0:


Loading weights:   0%|          | 0/378 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

EPOCH 0:


...

  0%|          | 0/597 [00:00<?, ?it/s]

Train Loss: 17.122272491455078
EPOCH 1:


  0%|          | 0/597 [00:00<?, ?it/s]

Train Loss: 11.301875114440918


(np.float64(0.7332953076904222), np.float64(0.011236604425156817))

In [17]:
model.backbone.save_pretrained("weights/")
torch.save(model.proj.state_dict(), "weights/proj.pt")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
import pickle

with open("fold_preds.pickle", "wb") as f:
    pickle.dump(fold_preds, f)